# Deep Backtest Analysis

Compare base vs advanced feature runs, inspect robustness, and write recommendations before any real-money decision.

In [1]:
import json
from pathlib import Path

import pandas as pd
import plotly.express as px

ROOT = Path("../data/processed/deep_backtests")
run_dir = sorted([p for p in ROOT.glob("*") if (p / "manifest.json").exists()])[-1]
manifest = json.loads((run_dir / "manifest.json").read_text())
summary = json.loads((run_dir / "analysis" / "summary.json").read_text())
run_dir, summary

IndexError: list index out of range

In [ ]:
summary_frame = pd.DataFrame(summary).T.reset_index(names="run")
summary_frame[["run", "rubric_grade", "go_no_go", "bet_count", "net_pnl", "brier_score", "sharpe"]]

In [ ]:
equity_frames = []
for path in sorted((run_dir / "runs").glob("*/equity_curve.parquet")):
    frame = pd.read_parquet(path)
    frame["run"] = path.parent.name
    equity_frames.append(frame)
equity = pd.concat(equity_frames, ignore_index=True) if equity_frames else pd.DataFrame()
px.line(
    equity, x="as_of", y="equity", color="run", title="Equity Curves"
) if not equity.empty else equity

In [ ]:
ablation = pd.read_csv(run_dir / "analysis" / "ablation.csv")
px.bar(
    ablation[
        ablation["metric"].isin(["net_pnl", "brier_score", "portfolio_sharpe", "portfolio_return"])
    ],
    x="metric",
    y="delta",
    color="run",
    title="Feature Ablation Deltas",
)

In [ ]:
calibration = pd.read_csv(run_dir / "analysis" / "calibration.csv")
px.scatter(
    calibration,
    x="model_prob_mean",
    y="outcome_rate",
    size="count",
    color="run",
    title="Reliability Diagram",
)

In [ ]:
failures = pd.read_csv(run_dir / "analysis" / "failures.csv")
failures.sort_values("pnl").head(10)

## Why We Failed The Rubric

Use this table to separate sample-size failures from weak-signal, miscalibration, and risk failures. Do not relax Go / No-Go constraints unless the failure mode is clearly operational rather than statistical.

In [ ]:
rubric_failures = pd.read_csv(run_dir / "analysis" / "rubric_failures.csv")
rubric_failures[
    [
        "run",
        "primary_failure_mode",
        "bet_count",
        "brier_score",
        "net_pnl",
        "constraints",
        "rubric_recommendation",
    ]
]

## Feature Diagnostics

Positive feature/outcome correlation and positive top-minus-bottom hit-rate indicate a potentially useful feature. Low correlation plus unstable signs across quarters is likely noise.

In [ ]:
feature_diagnostics = pd.read_csv(run_dir / "analysis" / "feature_diagnostics.csv")
feature_diagnostics.sort_values(["run", "feature_outcome_corr"], ascending=[True, False]).head(20)

In [ ]:
feature_stability = pd.read_csv(run_dir / "analysis" / "feature_stability.csv")
stable = feature_stability[feature_stability["quarter"].eq("all")]
stable.sort_values(
    ["run", "stability_score", "sign_flip_rate"], ascending=[True, False, True]
).head(20)

## Category / Regime Focus

Use this to decide whether to continue broadly or pivot to a narrower domain.

In [ ]:
by_category = pd.read_csv(run_dir / "analysis" / "by_category.csv")
regimes = pd.read_csv(run_dir / "analysis" / "regimes.csv")
display(by_category.sort_values(["run", "net_pnl"], ascending=[True, False]))
display(regimes.sort_values(["run", "net_pnl"], ascending=[True, False]).head(20))

## Phase 6 Recommendation

The recommendation table chooses between continuing iteration, narrowing the domain, or pausing based on PnL, category concentration, feature predictiveness, and rubric failure mode.

In [ ]:
recommendations = pd.read_csv(run_dir / "analysis" / "recommendations.csv")
recommendations

## Recommendations

- Promote only runs that are `Decision-Grade` and `Go` after reviewing sample size, significance, drawdown, and operational constraints.
- Keep LLM/on-chain features enabled only where ablation improves both calibration and economic value after cost.
- Tighten category-level thresholds where failure cases cluster.
- Re-run after adding fresh resolved markets; do not tune on this notebook and reuse the same holdout.